Config

In [1]:
import json, re, os, random
import numpy as np, pandas as pd
 
DATASET_PATH = "../data/tanglish.jsonl"
TRIPLES_PATH = "../data/parallel_triples.csv"
RESULTS_DIR  = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)
 
TOKENIZERS = {                      # tokenizers only, no model weights are downloaded
    "gemma-2b":      "google/gemma-2b",        # the model you fine-tuned
    "qwen2.5-0.5b":  "Qwen/Qwen2.5-0.5B",      # recent multilingual-ish small LM
    "xlm-roberta":   "xlm-roberta-base",       # multilingual, trained on Tamil
    "mt5":           "google/mt5-small",       # multilingual, trained on Tamil
    "gpt2":          "gpt2",                   # English-centric reference point
}
 
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]
outs = [r["output"].strip() for r in raw if r.get("output", "").strip()]
ins  = [r["instruction"].strip() for r in raw if r.get("instruction", "").strip()]
print(f"{len(outs)} Tanglish outputs | {len(ins)} English instructions")


613 Tanglish outputs | 613 English instructions


Load Tokenizers

In [2]:
from transformers import AutoTokenizer
 
toks = {}
for name, hub in TOKENIZERS.items():
    try:
        toks[name] = AutoTokenizer.from_pretrained(hub)
    except Exception as e:
        print("SKIPPED", name, "->", str(e)[:150])
print("Loaded:", {k: len(t) for k, t in toks.items()}, "(name: vocab size)")
 


/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
[transformers] Could not extract SentencePiece model from /home/prathoseraajv/.cache/huggingface/hub/models--google--mt5-small/snapshots/73fb5dbe4756edadc8fbe8c769b0a109493acf7a/spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment

SKIPPED mt5 -> `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.
Loaded: {'gemma-2b': 256000, 'qwen2.5-0.5b': 151665, 'xlm-roberta': 250002, 'gpt2': 50257} (name: vocab size)
